In [32]:
from __future__ import annotations

import argparse
import math
import warnings
from pathlib import Path
from typing import Literal, Sequence

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

BootstrapMethod = Literal["stratified", "moving_block"]


def _get_probability_columns(
    frame: pd.DataFrame,
    probability_cols: Sequence[str] | None,
) -> list[str]:
    """Return model probability columns, validating explicitly supplied names."""
    if probability_cols is None:
        columns = [
            col for col in frame.columns
            if isinstance(col, str) and col.endswith("_probability")
        ]
    else:
        columns = list(probability_cols)

    if not columns:
        raise ValueError(
            "No probability columns found. Expected columns ending in "
            "'_probability', or pass probability_cols explicitly."
        )
    if len(columns) != len(set(columns)):
        raise ValueError("probability_cols contains duplicate column names.")

    missing = [col for col in columns if col not in frame.columns]
    if missing:
        raise ValueError(f"Probability column(s) not found in CSV: {missing}")
    return columns


def _bootstrap_indices(
    y: np.ndarray,
    rng: np.random.Generator,
    method: BootstrapMethod,
    block_size: int,
) -> np.ndarray:
    """Draw paired row indices using the requested bootstrap scheme."""
    n = len(y)

    if method == "stratified":
        negative_idx = np.flatnonzero(y == 0)
        positive_idx = np.flatnonzero(y == 1)
        sampled_negative = rng.choice(negative_idx, size=len(negative_idx), replace=True)
        sampled_positive = rng.choice(positive_idx, size=len(positive_idx), replace=True)
        sampled = np.concatenate((sampled_negative, sampled_positive))
        rng.shuffle(sampled)
        return sampled

    if method == "moving_block":
        if not 1 <= block_size <= n:
            raise ValueError(
                f"For moving_block, block_size must be between 1 and {n}; "
                f"received {block_size}."
            )
        starts = np.arange(n - block_size + 1)
        n_blocks = math.ceil(n / block_size)
        selected_starts = rng.choice(starts, size=n_blocks, replace=True)
        return np.concatenate(
            [np.arange(start, start + block_size) for start in selected_starts]
        )[:n]

    raise ValueError("method must be either 'stratified' or 'moving_block'.")

In [33]:
def bootstrap_metrics_from_file(
    prediction_file: str | Path,
    *,
    actual_col: str = "actual",
    probability_cols: Sequence[str] | None = None,
    method: BootstrapMethod = "stratified",
    n_bootstrap: int = 5000,
    confidence_level: float = 0.95,
    block_size: int = 20,
    random_state: int = 42,
    date_col: str | None = "date",
    output_file: str | Path | None = None,
    model_prefix: str = "",
) -> pd.DataFrame:
    """Calculate ROC AUC, Average Precision (AP), and bootstrap percentile CIs."""
    prediction_file = Path(prediction_file)
    if not prediction_file.is_file():
        raise FileNotFoundError(f"Prediction CSV not found: {prediction_file}")
    if n_bootstrap <= 0:
        raise ValueError("n_bootstrap must be a positive integer.")
    if not 0.0 < confidence_level < 1.0:
        raise ValueError("confidence_level must be strictly between 0 and 1.")
    if method not in ("stratified", "moving_block"):
        raise ValueError("method must be either 'stratified' or 'moving_block'.")

    frame = pd.read_csv(prediction_file)
    if actual_col not in frame.columns:
        raise ValueError(
            f"Label column '{actual_col}' not found. Available columns: "
            f"{list(frame.columns)}"
        )

    probability_cols = _get_probability_columns(frame, probability_cols)

    # Preserve temporal order for moving-block resampling.
    if method == "moving_block" and date_col is not None and date_col in frame.columns:
        frame = frame.copy()
        frame[date_col] = pd.to_datetime(frame[date_col], errors="raise")
        frame = frame.sort_values(date_col, kind="stable").reset_index(drop=True)

    y_numeric = pd.to_numeric(frame[actual_col], errors="coerce").to_numpy(dtype=float)
    if not np.all(np.isfinite(y_numeric)) or not np.all(np.isin(y_numeric, [0, 1])):
        raise ValueError(
            f"'{actual_col}' must contain only finite binary labels 0 and 1, "
            "with no missing values."
        )
    y = y_numeric.astype(np.int8)

    n = len(y)
    n_negative = int(np.sum(y == 0))
    n_positive = int(np.sum(y == 1))
    if n_negative == 0 or n_positive == 0:
        raise ValueError(
            "The input test set must contain both classes; "
            f"found 0={n_negative}, 1={n_positive}."
        )
    if method == "moving_block" and not 1 <= block_size <= n:
        raise ValueError(
            f"For moving_block, block_size must be between 1 and {n}; "
            f"received {block_size}."
        )

    probabilities: dict[str, np.ndarray] = {}
    for column in probability_cols:
        values = pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)
        if not np.all(np.isfinite(values)):
            raise ValueError(
                f"Probability column '{column}' contains missing or non-finite values."
            )
        if np.any((values < 0.0) | (values > 1.0)):
            raise ValueError(
                f"Probability column '{column}' contains values outside [0, 1]."
            )
        probabilities[column] = values

    point_estimates: dict[str, tuple[float, float]] = {}
    for column, probs in probabilities.items():
        point_estimates[column] = (
            float(roc_auc_score(y, probs)),
            float(average_precision_score(y, probs)),
        )

    rng = np.random.default_rng(random_state)
    bootstrap_values = {
        column: {"ROC_AUC": [], "AP": []}
        for column in probability_cols
    }

    attempts = 0
    max_attempts = n_bootstrap if method == "stratified" else max(n_bootstrap * 20, n_bootstrap)
    while attempts < max_attempts:
        if all(
            len(bootstrap_values[col]["ROC_AUC"]) >= n_bootstrap
            for col in probability_cols
        ):
            break
        attempts += 1
        indices = _bootstrap_indices(y, rng, method, block_size)
        y_boot = y[indices]

        if np.unique(y_boot).size < 2:
            continue

        for column, probs in probabilities.items():
            p_boot = probs[indices]
            bootstrap_values[column]["ROC_AUC"].append(
                float(roc_auc_score(y_boot, p_boot))
            )
            bootstrap_values[column]["AP"].append(
                float(average_precision_score(y_boot, p_boot))
            )

    alpha = 1.0 - confidence_level
    lower_pct = 100.0 * alpha / 2.0
    upper_pct = 100.0 * (1.0 - alpha / 2.0)
    rows: list[dict[str, object]] = []

    for column in probability_cols:
        auc_values = np.asarray(bootstrap_values[column]["ROC_AUC"], dtype=float)
        ap_values = np.asarray(bootstrap_values[column]["AP"], dtype=float)
        valid = min(len(auc_values), len(ap_values))

        if valid == 0:
            raise RuntimeError(
                "No valid bootstrap replicates were generated. For moving_block, "
                "try a smaller block_size or use method='stratified'."
            )
        if valid < n_bootstrap:
            warnings.warn(
                f"Model '{column}': only {valid}/{n_bootstrap} valid replicates "
                f"were generated after {attempts} attempts. CI estimates may be noisy.",
                RuntimeWarning,
            )

        auc_point, ap_point = point_estimates[column]
        model_name = column.removesuffix("_probability")
        if model_prefix:
            model_name = f"{model_prefix}_{model_name}"

        rows.append({
            "Model": model_name,
            "N": n,
            "Positives": n_positive,
            "Negatives": n_negative,
            "Prevalence": n_positive / n,
            "Bootstrap_Method": method,
            "ROC_AUC": auc_point,
            "ROC_AUC_LOW": float(np.percentile(auc_values, lower_pct)),
            "ROC_AUC_HIGH": float(np.percentile(auc_values, upper_pct)),
            "AP": ap_point,
            "AP_LOW": float(np.percentile(ap_values, lower_pct)),
            "AP_HIGH": float(np.percentile(ap_values, upper_pct)),
            "Bootstrap_Valid": valid,
            "Bootstrap_Attempts": attempts,
        })

    summary = pd.DataFrame(rows).sort_values(
        "ROC_AUC", ascending=False, na_position="last"
    ).reset_index(drop=True)

    if output_file is not None:
        output_path = Path(output_file)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        summary.to_csv(output_path, index=False)

    return summary

In [34]:
Market = "SP500"

MARKET_LIST = ["SP500", "STOXX", "CAC", "DAX", "FTSE", "CBU0"]
OUTPUT_DIR = Path("Results/Benchmark_Outputs/GJR_GARCH")

# Define paths for your standard ML benchmark predictions and GARCH predictions
ml_pred_file = f"Results/Benchmark_Outputs/With_news_v2/{Market}_finderBenchmark_test_predictions.csv"
garch_pred_file = f"Results/Benchmark_Outputs/GJR_GARCH/{Market}_GJR_GARCH_test_predictions.csv" # Update market/filename as needed

master_output_path = Path(f"Results/Benchmark_Outputs/{Market}_Combined_ML_GARCH_BootstrapSummary_news.csv")

summaries = []

# 1. Evaluate Standard ML Models (typically using 'stratified' bootstrap)
if Path(ml_pred_file).is_file():
    print(f"Processing ML Benchmark models from: {ml_pred_file}")
    ml_summary = bootstrap_metrics_from_file(
        ml_pred_file,
        method="moving_block",
        n_bootstrap=10000,
        block_size=21,
        confidence_level=0.95,
        random_state=42,
        model_prefix="ML"
    )
    summaries.append(ml_summary)
else:
    print(f"ML prediction file not found: {ml_pred_file}")

# 2. Evaluate GARCH Model (typically using 'moving_block' for time series, or 'stratified')
if Path(garch_pred_file).is_file():
    print(f"Processing GARCH model from: {garch_pred_file}")
    garch_summary = bootstrap_metrics_from_file(
        garch_pred_file,
        method="moving_block",  # Recommended default for GARCH time-series[cite: 2]
        n_bootstrap=10000,
        confidence_level=0.95,
        block_size=21,
        random_state=42,
        model_prefix="GARCH"
    )
    summaries.append(garch_summary)
else:
    print(f"GARCH prediction file not found: {garch_pred_file}")

# 3. Combine, Sort by ROC AUC, and Export Final Master File
if summaries:
    master_summary = pd.concat(summaries, ignore_index=True)
    master_summary = master_summary.sort_values(
        "ROC_AUC", ascending=False, na_position="last"
    ).reset_index(drop=True)
    
    master_output_path.parent.mkdir(parents=True, exist_ok=True)
    master_summary.to_csv(master_output_path, index=False)
    
    print("\n" + "="*80)
    print("FINAL COMBINED MODEL PERFORMANCE SUMMARY (ML & GARCH)")
    print("="*80)
    print(master_summary.to_string(index=False, float_format=lambda val: f"{val:.4f}"))
    print(f"\nSuccessfully saved combined summary to: {master_output_path.resolve()}")
else:
    print("No prediction files were found to process.")

Processing ML Benchmark models from: Results/Benchmark_Outputs/With_news_v2/SP500_finderBenchmark_test_predictions.csv
Processing GARCH model from: Results/Benchmark_Outputs/GJR_GARCH/SP500_GJR_GARCH_test_predictions.csv

FINAL COMBINED MODEL PERFORMANCE SUMMARY (ML & GARCH)
      Model   N  Positives  Negatives  Prevalence Bootstrap_Method  ROC_AUC  ROC_AUC_LOW  ROC_AUC_HIGH     AP  AP_LOW  AP_HIGH  Bootstrap_Valid  Bootstrap_Attempts
GARCH_garch 376         17        359      0.0452     moving_block   0.7516       0.5130        0.8551 0.1590  0.0461   0.2425            10000               10000
    ML_LSTM 376         17        359      0.0452     moving_block   0.6099       0.4728        0.7619 0.0630  0.0297   0.1270            10000               10000
ML_Logistic 376         17        359      0.0452     moving_block   0.6031       0.3161        0.7363 0.0756  0.0188   0.1344            10000               10000

Successfully saved combined summary to: /Users/ida/Library/Mobile D